# N4E / 9D — non-ReMap Part 2 canonical promotion summary

Read-only decision notebook for the non-ReMap Part 2 / source-native official KG lane.

Scope: summarize the canonical-promotion candidate and decision for non-ReMap tranches after validation card `t_61fabcf3` and reviewer gate `t_ce6e158c`.

Out of scope: ReMap `tf_binds_enhancer`, which remains a separate/deferred path and is not included in this promotion lane.

## Safety contract

Default behavior is safe, local, and read-only:

- no canonical writes;
- no GCS writes;
- no downloads;
- no full Parquet scans;
- local report/manifest reads only;
- any optional heavy or remote check must be guarded by an explicit environment flag.

This notebook is a narrative and audit surface. Promotion itself belongs in the gated promotion card (`t_17cfc462`) after reviewer approval and fresh destination rechecks.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT.parent / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "manage_db").exists():
    raise RuntimeError(f"cannot locate Jouvence repository root from {Path.cwd()}")

REPORT_MD = REPO_ROOT / "docs" / "non_remap_part2_canonical_promotion_validation_report.md"
REPORT_JSON = REPO_ROOT / "docs" / "non_remap_part2_canonical_promotion_validation.json"  # optional current docs-side JSON, if exported
PROMOTION_JSON = REPO_ROOT / "docs" / "part2_biogrid_ppi_canonical_promotion_manifest.json"  # optional current docs-side JSON, if exported
PROMOTION_REPORT_MD = REPO_ROOT / "docs" / "part2_biogrid_ppi_canonical_promotion_report.md"

ALLOW_HEAVY_SCAN = os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION") == "1"
ALLOW_REMOTE = os.environ.get("TXGNN_NOTEBOOK_REMOTE_CHECKS") == "1"

print("REPO_ROOT:", REPO_ROOT)
print("validation report:", REPORT_MD, "exists=", REPORT_MD.exists())
print("validation JSON:", REPORT_JSON, "exists=", REPORT_JSON.exists())
print("t_17cfc462 promotion manifest:", PROMOTION_JSON, "exists=", PROMOTION_JSON.exists())
print("t_17cfc462 promotion report:", PROMOTION_REPORT_MD, "exists=", PROMOTION_REPORT_MD.exists())
print("heavy scans enabled:", ALLOW_HEAVY_SCAN)
print("remote checks enabled:", ALLOW_REMOTE)

## Promotion lane cards

This lane deliberately separates validation, review approval, and canonical write execution.

In [ ]:
pd.DataFrame([
    {
        "stage": "validation",
        "task": "t_61fabcf3",
        "status": "complete",
        "artifact": "docs/non_remap_part2_canonical_promotion_validation_report.md; optional docs/non_remap_part2_canonical_promotion_validation.json",
        "canonical_writes": False,
    },
    {
        "stage": "review gate",
        "task": "t_ce6e158c",
        "status": "approved",
        "artifact": "Kanban handoff-approved manifest; local manifest file optional/pending",
        "canonical_writes": False,
    },
    {
        "stage": "promotion execution",
        "task": "t_17cfc462",
        "status": "complete for approved BioGRID physical PPI tranche; post-promotion manifest/report detected locally",
        "artifact": "docs/part2_biogrid_ppi_canonical_promotion_report.md; optional docs/part2_biogrid_ppi_canonical_promotion_manifest.json",
        "canonical_writes": "performed only by explicit promotion card t_17cfc462; this notebook remains read-only",
    },
    {
        "stage": "separate deferred lane",
        "task": "t_3479936e and ReMap descendants/reviewers",
        "status": "not included here",
        "artifact": "ReMap tf_binds_enhancer staging/review artifacts",
        "canonical_writes": False,
    },
])

## Final decision matrix

Decision vocabulary:

- `promote_now`: approved for the non-ReMap canonical write path.
- `include_as_feature_context`: useful as feature/context/review material, not a canonical biological edge/evidence promotion here.
- `defer`: staged or plausible, but blocked by schema, endpoint policy, license/mapping, bounded validation, or separate promotion gate.
- `reject`: do not promote; retain only as rejected/candidate diagnostics if useful.

Reviewer gate `t_ce6e158c` approved only BioGRID physical PPI for canonical promotion. No reviewer canonical writes were performed.

In [ ]:
decision_rows = [
    {
        "tranche": "BioGRID physical PPI",
        "decision": "promote_now",
        "review_cards": "t_28f83a7b; t_d64b99c0; gate t_ce6e158c",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/",
        "staged_files": "biogrid-categorized-20260622/edges/protein_interacts_protein.parquet; biogrid-categorized-20260622/evidence/protein_interacts_protein.parquet",
        "canonical_destination": "gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet; gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet",
        "rows_or_counts": "3,550 edges; 12,288 evidence; duplicate_edges=0; edges_without_evidence=0; evidence_without_edges=0; endpoint anti-joins=0",
        "reason": "Active schema relation; protein endpoints anti-join clean against cached canonical protein ids/uniprot_ids; edge/evidence support clean. Preserve evidence_class; do not infer protein_complex nodes from pairwise/system rows.",
    },
    {
        "tranche": "IntAct corrected/bounded protein_interacts_protein",
        "decision": "defer",
        "review_cards": "t_100231b1; t_0964be36",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/intact-protein-interactions-policy-fixed/runs/20260622T122314Z-bounded100k/",
        "staged_files": "edges/protein_interacts_protein.parquet; evidence/protein_interacts_protein.parquet; evidence/protein_interacts_protein_negative.parquet; reports/rejected_rows.parquet",
        "canonical_destination": "same protein_interacts_protein canonical edge/evidence paths if separately approved",
        "rows_or_counts": "34,515 edges; 46,425 evidence; bounded sample; 496 negative evidence; 53,575 rejected rows",
        "reason": "Bounded recovery staging and no canonical node-root endpoint anti-join; not canonical-grade for this promotion despite clean support checks.",
    },
    {
        "tranche": "BioGRID protein_has_ptm_site / ptm_site split",
        "decision": "defer",
        "review_cards": "t_28f83a7b; t_d64b99c0",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/",
        "staged_files": "biogrid-categorized-20260622/nodes/ptm_site.parquet; edges/protein_has_ptm_site.parquet; evidence/protein_has_ptm_site.parquet",
        "canonical_destination": "gs://jouvencekb/kg/v2/nodes/ptm_site.parquet; gs://jouvencekb/kg/v2/edges/protein_has_ptm_site.parquet; gs://jouvencekb/kg/v2/evidence/protein_has_ptm_site.parquet",
        "rows_or_counts": "28,169 ptm_site nodes/edges; 62,096 evidence; support/endpoint checks pass",
        "reason": "ptm_site/protein_has_ptm_site absent from active schema; PSI-MOD/isoform handling needs future review.",
    },
    {
        "tranche": "miRNA real-source alias/target path",
        "decision": "defer",
        "review_cards": "t_f1b51a59; t_1734823c; t_95bbd18c; t_08770b04",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/mirna-targets-real/",
        "staged_files": "nodes/mirna.parquet; edges/mirna_targets_gene.parquet; edges/mirna_precursor_produces_mature_mirna.parquet; evidence/mirna_targets_gene.parquet",
        "canonical_destination": "mirna node/edge/evidence canonical paths after schema naming approval",
        "rows_or_counts": "3,929 miRNA nodes; 351,958 miRNA→gene edges; 1,707 precursor edges; 868,896 evidence",
        "reason": "miRNA node type and relation naming absent/mismatched in active schema; defer until schema approval.",
    },
    {
        "tranche": "lncRNA GENCODE nodes",
        "decision": "defer",
        "review_cards": "t_35dddc93; t_e3e2a5a0",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622/",
        "staged_files": "nodes/lncrna.parquet; reports/gencode_lncrna_gene_summary.parquet",
        "canonical_destination": "gs://jouvencekb/kg/v2/nodes/lncrna.parquet",
        "rows_or_counts": "197,208 lncRNA nodes; 37,576 summary rows",
        "reason": "lncrna node type absent and canonical ID policy unresolved.",
    },
    {
        "tranche": "LncRNADisease disease edges/candidates",
        "decision": "defer",
        "review_cards": "t_35dddc93; t_e3e2a5a0",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622/",
        "staged_files": "candidates/lncrna_associated_disease.parquet; edges/lncrna_associated_disease.parquet; reports/rejected_or_needs_review.parquet",
        "canonical_destination": "gs://jouvencekb/kg/v2/edges/lncrna_associated_disease.parquet; gs://jouvencekb/kg/v2/evidence/lncrna_associated_disease.parquet",
        "rows_or_counts": "0 active edges; 6,598 candidate/rejected rows",
        "reason": "Disease edge rows gated by license/mapping/schema; do not promote here.",
    },
    {
        "tranche": "RBP/RNA CLIP ENCORI/POSTAR context",
        "decision": "include_as_feature_context",
        "review_cards": "t_89b3ddaf; t_010bc1e4",
        "staged_prefix": "gs://jouvencekb/kg/v2/staging/rbp-rna-clip-encori-pilot-20260622T135045Z/",
        "staged_files": "reports/candidate_clip_evidence.parquet; reports/rejected_rows.parquet; empty edge/evidence files",
        "canonical_destination": "no canonical edge/evidence path recommended from this validation",
        "rows_or_counts": "0 active edges/evidence; 100 candidates; 100 rejected/context rows",
        "reason": "Missing source-native endpoints / lncRNA schema support; context only.",
    },
    {
        "tranche": "Expression/coexpression feature-context contract",
        "decision": "include_as_feature_context",
        "review_cards": "t_c8f1dbc0; t_a2820a4e",
        "staged_prefix": "feature/context reports only",
        "staged_files": "feature/context contract artifacts",
        "canonical_destination": "feature/context namespace only; no canonical edge path recommended",
        "rows_or_counts": "not a canonical edge tranche",
        "reason": "Reviewed as feature/context contract only; no approved active causal/mechanistic relation or raw licensed source package.",
    },
    {
        "tranche": "HPA cellular_component / protein localization",
        "decision": "defer",
        "review_cards": "t_4bda37e9; t_51714eaf; t_41852a2b",
        "staged_prefix": "gs://jouvencekb/kg/staging/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/",
        "staged_files": "nodes/cellular_component.parquet; edges/protein_located_in_cellular_component.parquet; evidence/protein_located_in_cellular_component.parquet; cellular_component subtype files",
        "canonical_destination": "cellular_component node and localization edge/evidence canonical paths after schema/policy approval",
        "rows_or_counts": "60 cellular_component nodes; 276,356 localization edges; 536,967 localization evidence; 14 subtype edges/evidence",
        "reason": "Active schema lacks cellular_component/protein localization elements; all-ENSP vs canonical-ENSP policy unresolved.",
    },
    {
        "tranche": "Complex Portal protein_complex",
        "decision": "defer",
        "review_cards": "t_9d94edb6; t_c34d9545",
        "staged_prefix": "gs://jouvencekb/kg/staging/complex-portal-protein-complexes-2026-06-22/",
        "staged_files": "nodes/protein_complex.parquet; membership/nested edge/evidence files; mappings/complex_portal_participants_rejected.parquet",
        "canonical_destination": "protein_complex node and membership edge/evidence canonical paths after schema/name approval",
        "rows_or_counts": "2,498 complexes; 625 protein membership edges/evidence; 117 nested complex edges/evidence; 9,359 rejected participants",
        "reason": "protein_complex node and relation names absent from active schema; UniProt isoform policy unresolved.",
    },
    {
        "tranche": "UniProt PTM sites",
        "decision": "defer",
        "review_cards": "t_ef541e0e; t_13a70788",
        "staged_prefix": "gs://jouvencekb/kg/staging/uniprot-ptm-sites-2026-06-22/",
        "staged_files": "nodes/ptm_site.parquet; edges/protein_has_ptm_site.parquet; evidence/protein_has_ptm_site.parquet; diagnostics/ptm_site_disease_link_candidates.parquet",
        "canonical_destination": "ptm_site/protein_has_ptm_site canonical paths after schema approval",
        "rows_or_counts": "92,578 ptm_site nodes/edges; 173,491 evidence; 8 disease-link diagnostics",
        "reason": "ptm_site/protein_has_ptm_site absent from active schema; PSI-MOD/isoform handling future review.",
    },
    {
        "tranche": "OpenTargets/Ensembl gene_paralog_gene",
        "decision": "defer",
        "review_cards": "t_e7603b95; t_b19c67f8",
        "staged_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/gene_paralog_gene/opentargets-target-homologues-20260622/",
        "staged_files": "edges/gene_paralog_gene.parquet; evidence/gene_paralog_gene.parquet",
        "canonical_destination": "gs://jouvencekb/kg/v2/edges/gene_paralog_gene.parquet; gs://jouvencekb/kg/v2/evidence/gene_paralog_gene.parquet",
        "rows_or_counts": "3,544,825 edges; 3,544,825 evidence",
        "reason": "gene_paralog_gene absent from active schema; source-order/symmetry/evidence-schema gate unresolved.",
    },
    {
        "tranche": "Pharmacology/protein-native accepted staged batches",
        "decision": "defer",
        "review_cards": "t_19516b59; t_ee55140a",
        "staged_prefix": "staged/non-canonical producer outputs from replacement review",
        "staged_files": "molecule_targets_protein; disease_associated_protein; pathway_contains_protein; molecule_synergizes_molecule; molecule_treats_disease edge/evidence batches",
        "canonical_destination": "relation-specific canonical paths only after separate explicit apply/promotion card",
        "rows_or_counts": "accepted as staged/non-canonical; counts belong to producer/review cards",
        "reason": "Separate explicit apply/promotion card required; source-control/patch hygiene gate recorded.",
    },
    {
        "tranche": "Sci-Plex cell_type_responds_to_molecule candidate/context downgrade",
        "decision": "include_as_feature_context",
        "review_cards": "t_587ab15a",
        "staged_prefix": "gs://jouvencekb/kg/staging/cell_type_molecule_candidate_context_sciplex2_20260622_t_63ca49a0/",
        "staged_files": "candidates/cell_type_responds_to_molecule_sciplex2_candidates.parquet; rejected candidates; empty edge/evidence files",
        "canonical_destination": "no canonical edge/evidence path recommended from this validation",
        "rows_or_counts": "0 active edges/evidence; 14 candidates; 25 rejected/context rows; 1 cell_type node; 2 molecule nodes",
        "reason": "Source obs lacks response/effect metric versus control; retain only context/candidates.",
    },
    {
        "tranche": "Metadata/features/source coverage batch",
        "decision": "include_as_feature_context",
        "review_cards": "t_67465eac; t_4de0dfbc",
        "staged_prefix": "feature/context/report namespace",
        "staged_files": "source coverage reports/features",
        "canonical_destination": "no canonical edge path recommended by this validation",
        "rows_or_counts": "staged/review artifacts; not edge promotion material",
        "reason": "Feature/provenance review material, not biological KG edge/evidence promotion.",
    },
    {
        "tranche": "ReMap tf_binds_enhancer active/full/all-chromosome cards",
        "decision": "defer_separate_not_included",
        "review_cards": "t_3479936e; t_17f2b3d5; t_8fff8356; t_6d8e46c8; t_5738004a",
        "staged_prefix": "ReMap staging prefixes under remap-tf-binds-enhancer / all-chromosome cards",
        "staged_files": "tf_binds_enhancer edge/evidence staging artifacts when separately accepted",
        "canonical_destination": "gs://jouvencekb/kg/v2/edges/tf_binds_enhancer.parquet; gs://jouvencekb/kg/v2/evidence/tf_binds_enhancer.parquet",
        "rows_or_counts": "not evaluated for this non-ReMap promotion",
        "reason": "Explicitly excluded by human decision; ReMap must not block or be silently included in this lane.",
    },
]

decision_df = pd.DataFrame(decision_rows)
decision_df

## Approved canonical promotion manifest

The approved manifest from reviewer gate `t_ce6e158c` contains exactly one `promote_now` tranche: BioGRID physical PPI.

Important caveats to preserve:

- relation `protein_interacts_protein` is active in `manage_db/kg_schema.py`;
- do not promote BioGRID PTM files in this tranche;
- evidence includes complex/cofractionation-associated physical evidence classes, but this does **not** create protein-complex nodes or membership edges;
- canonical destination files did not exist during reviewer probe, but promotion execution must recheck destinations immediately before writing.

In [ ]:
approved_manifest = {
    "canonical_root": "gs://jouvencekb/kg/v2",
    "canonical_writes_performed_by_reviewer": False,
    "approved_tranches": [
        {
            "tranche": "BioGRID physical PPI",
            "decision": "promote_now",
            "relations": ["protein_interacts_protein"],
            "nodes": [],
            "features": [],
            "staged_files": {
                "edges": "gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/biogrid-categorized-20260622/edges/protein_interacts_protein.parquet",
                "evidence": "gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/biogrid-categorized-20260622/evidence/protein_interacts_protein.parquet",
            },
            "canonical_destination_paths": {
                "edges": "gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet",
                "evidence": "gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet",
            },
            "expected_counts": {
                "edges": 3550,
                "evidence": 12288,
                "duplicate_edges": 0,
                "edges_without_evidence": 0,
                "evidence_without_edges": 0,
                "x_endpoint_missing_id_or_uniprot": 0,
                "y_endpoint_missing_id_or_uniprot": 0,
                "organism_pair_counts": {"9606/9606": 12288},
                "evidence_class_counts": {
                    "binary_physical": 1615,
                    "biochemical_or_ptm_like_activity": 372,
                    "complex_or_cofractionation_association": 10301,
                },
            },
            "source_review_tasks": ["t_28f83a7b", "t_d64b99c0"],
            "caveats": [
                "Do not promote BioGRID PTM files in this tranche because ptm_site/protein_has_ptm_site are not active schema elements.",
                "Preserve evidence_class and do not infer protein_complex nodes or complex membership from BioGRID TAB3 pairwise rows.",
                "Recheck canonical destinations immediately before promotion to avoid overwriting newly created files.",
            ],
        }
    ],
}

pd.json_normalize(approved_manifest["approved_tranches"])

## Local validation and t_17cfc462 promotion evidence

These cells read existing local reports only. The promotion execution artifacts are the `t_17cfc462` BioGRID physical PPI report and manifest:

- `docs/part2_biogrid_ppi_canonical_promotion_report.md`
- optional current `docs/part2_biogrid_ppi_canonical_promotion_manifest.json` if exported; older `.omoc/reports/...` copies are historical/non-default only

When present, the notebook summarizes promoted canonical paths, row counts, readback SHA256 checksums, no-ReMap write assertion, rollback notes, and targeted validation status. It does not perform canonical/GCS writes or shell/GCS calls.

In [ ]:
def load_json(path: Path) -> dict[str, Any] | list[Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}", "path": str(path)}


def read_text_excerpt(path: Path, max_chars: int = 2000) -> str | None:
    if not path.exists():
        return None
    text = path.read_text()
    return text[:max_chars] + ("\n... [truncated]" if len(text) > max_chars else "")


validation_payload = load_json(REPORT_JSON)
promotion_payload = load_json(PROMOTION_JSON)
promotion_report_excerpt = read_text_excerpt(PROMOTION_REPORT_MD)


def promotion_evidence_summary(payload: dict[str, Any] | list[Any] | None) -> dict[str, Any]:
    if not isinstance(payload, dict):
        return {
            "promotion_manifest_exists": payload is not None,
            "promotion_report_exists": PROMOTION_REPORT_MD.exists(),
        }
    relation = payload.get("relation")
    relation_report = (payload.get("audit_edge_evidence", {})
        .get("relation_reports", {})
        .get(relation, {})) if relation else {}
    return {
        "promotion_manifest_exists": True,
        "promotion_report_exists": PROMOTION_REPORT_MD.exists(),
        "task_id": payload.get("task_id"),
        "run_id": payload.get("run_id"),
        "relation": relation,
        "canonical_paths_written": payload.get("canonical_paths_written"),
        "canonical_staging_prefix": payload.get("canonical_staging_prefix"),
        "source_staged_paths": payload.get("source_staged_paths"),
        "observed_counts": payload.get("observed_counts"),
        "audit_edge_evidence": relation_report,
        "sha256_canonical_readback": payload.get("sha256", {}).get("canonical_readback"),
        "sha256_matches_staged": payload.get("sha256", {}).get("matches"),
        "no_remap_writes": payload.get("no_remap_writes"),
        "rollback_notes": payload.get("rollback_notes"),
        "targeted_validation": payload.get("tests"),
    }

summary = {
    "validation_json_exists": validation_payload is not None,
    "validation_verdict": (validation_payload or {}).get("verdict") if isinstance(validation_payload, dict) else None,
    "canonical_promotion_performed_by_validation": (validation_payload or {}).get("canonical_promotion_performed") if isinstance(validation_payload, dict) else None,
    "promotion_evidence": promotion_evidence_summary(promotion_payload),
}
summary

In [ ]:
if validation_payload is None:
    print("Validation JSON is pending:", REPORT_JSON)
else:
    # Print only bounded top-level context; avoid dumping the 300KB+ JSON into notebook outputs by default.
    top_keys = sorted(validation_payload.keys()) if isinstance(validation_payload, dict) else []
    print("validation JSON keys:", top_keys)
    if isinstance(validation_payload, dict):
        for key in ["canonical_promotion_performed", "verdict", "recommendation", "recommended_promotion_set"]:
            if key in validation_payload:
                print(f"{key}:", validation_payload[key])

if promotion_payload is None:
    print("t_17cfc462 promotion manifest pending:", PROMOTION_JSON)
else:
    print("t_17cfc462 promotion manifest summary:")
    print(json.dumps(promotion_evidence_summary(promotion_payload), indent=2))

if promotion_report_excerpt is None:
    print("t_17cfc462 promotion report pending:", PROMOTION_REPORT_MD)
else:
    print("t_17cfc462 promotion report excerpt:")
    print(promotion_report_excerpt)

## Validation commands and post-promotion checks

Validation already run by `t_61fabcf3` (read-only):

```bash
uv run python scripts/validate_non_remap_part2_promotion.py   --json-out docs/non_remap_part2_canonical_promotion_validation.json   --markdown-out docs/non_remap_part2_canonical_promotion_validation_report.md
```

Notebook-level validation:

```bash
uv run python - <<'PYVALIDATE'
import nbformat
nb = nbformat.read('reproduce/13_non_remap_canonical_promotion_summary.ipynb', as_version=4)
nbformat.validate(nb)
print('nbformat.validate PASS', len(nb.cells), 'cells')
PYVALIDATE
uv run python - <<'PY'
import nbformat
from nbclient import NotebookClient
nb = nbformat.read('reproduce/13_non_remap_canonical_promotion_summary.ipynb', as_version=4)
NotebookClient(nb, timeout=120, kernel_name='python3').execute()
print('executed', len(nb.cells), 'cells')
PY
```

Promotion card `t_17cfc462` produced post-promotion evidence summarized from `docs/part2_biogrid_ppi_canonical_promotion_report.md` and, if exported, `docs/part2_biogrid_ppi_canonical_promotion_manifest.json`. Older `.omoc/reports/...` copies are historical/non-default only. The notebook reads current docs artifacts and summarizes:

- canonical paths written for BioGRID physical PPI only (`edges/protein_interacts_protein.parquet` and `evidence/protein_interacts_protein.parquet` under `gs://jouvencekb/kg/v2`);
- canonical row-count readback parity (`3,550` edges, `12,288` evidence);
- `manage_db.audit_edge_evidence` targeted validation with `ok=true`, `edges_without_evidence = 0`, and `evidence_without_edge = 0`;
- canonical readback SHA256 checksums matching staged local hashes;
- `no_remap_writes = true` and rollback limited to the two promoted BioGRID PPI canonical objects if reviewer rejects.

The decision matrix remains unchanged in substance: only BioGRID physical PPI is `promote_now`; ReMap `tf_binds_enhancer` and the other non-ReMap tranches remain deferred or feature-context-only.

In [ ]:
post_promotion_checklist = pd.DataFrame([
    {"check": "destination recheck", "expected": "canonical BioGRID PPI paths absent or explicitly safe to create; never blind overwrite"},
    {"check": "edge row count", "expected": 3550},
    {"check": "evidence row count", "expected": 12288},
    {"check": "duplicate_edges", "expected": 0},
    {"check": "edges_without_evidence", "expected": 0},
    {"check": "evidence_without_edges", "expected": 0},
    {"check": "endpoint anti-joins", "expected": "x=0, y=0 against canonical protein id/uniprot ids"},
    {"check": "evidence class distribution", "expected": "binary_physical=1615; biochemical_or_ptm_like_activity=372; complex_or_cofractionation_association=10301"},
    {"check": "ReMap exclusion", "expected": "no tf_binds_enhancer copy/write in this lane"},
])
post_promotion_checklist

## Official feature-layer distinction

A separate ReMap-independent node-feature wave exists for sequence-like node sequences and textual descriptions. Those cards (`t_b5dd2399`, `t_f9ef6389`, `t_76c42ace`, `t_1646af09`, `t_df83345a`, `t_61304e4a`, `t_e6227487`, `t_c1feb247`) are feature-layer candidates, not canonical biological edge/evidence promotions.

Feature-layer inclusion can be official without implying a canonical biological relation was promoted. In this notebook, `include_as_feature_context` means “keep as reviewed feature/context material” and not “write to `kg/v2/edges` or `kg/v2/evidence`”.

In [ ]:
feature_layer_rows = [
    {"feature_wave": "node sequence features", "cards": "t_b5dd2399; t_f9ef6389; t_df83345a; t_61304e4a; t_e6227487; t_c1feb247", "lane": "official feature layer candidate", "canonical_edge_promotion": False},
    {"feature_wave": "textual descriptions / summary features", "cards": "t_76c42ace; t_1646af09; t_df83345a; t_61304e4a; t_e6227487; t_c1feb247", "lane": "official feature layer candidate", "canonical_edge_promotion": False},
    {"feature_wave": "expression/coexpression context", "cards": "t_c8f1dbc0; t_a2820a4e", "lane": "feature/context only", "canonical_edge_promotion": False},
    {"feature_wave": "metadata/source coverage context", "cards": "t_67465eac; t_4de0dfbc", "lane": "feature/context/report namespace", "canonical_edge_promotion": False},
]
pd.DataFrame(feature_layer_rows)

## ReMap final interpretation

Use this notebook to keep the story honest:

- Non-ReMap tranches can be validated/reviewed/promoted first.
- ReMap `tf_binds_enhancer` remains separate/deferred until its own provenance dictionary, motif-backed evidence summaries, resource profile, and reviewer gate are accepted.
- A canonical KG promotion from this lane must not contain `tf_binds_enhancer` unless a later ReMap-specific promotion card explicitly writes it.
- BioGRID physical PPI is the only reviewer-approved non-ReMap Part 2 canonical promotion candidate summarized here.